# Example: Using Trained Random Forest Models

This notebook demonstrates how to:
- Load the trained models from `./model/`
- Make predictions on new data
- Handle raw input with flexible preprocessing
- Evaluate model performance

**Models included:**
- `rf_etiquette`: Classification model predicting energy labels (A-G)
- `rf_value`: Regression model predicting energy consumption (kWh/m²)

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
import warnings
import os

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported")

✓ Libraries imported


## 2. Load Pre-trained Models

In [ ]:
model_dir = Path('./.model')

# Load models
print("Loading models from ./model/")
print("=" * 70)

rf_etiquette = joblib.load(model_dir / 'rf_etiquette.joblib')
print(f"Loaded rf_etiquette")
print(f"  Classes: {rf_etiquette.classes_}")

rf_value = joblib.load(model_dir / 'rf_value.joblib')
print(f"Loaded rf_value")

# Load metadata
label_encoders = joblib.load(model_dir / 'label_encoders.joblib')
print(f"Loaded label_encoders: {len(label_encoders)} categorical columns")

feature_columns = joblib.load(model_dir / 'feature_columns.joblib')
print(f"Loaded feature_columns: {len(feature_columns)} features")

print("\n" + "=" * 70)
print("All models loaded successfully!")

Loading models from ./model/
✓ Loaded rf_etiquette
  Classes: ['A' 'B' 'C' 'D' 'E' 'F' 'G']
✓ Loaded rf_value
✓ Loaded label_encoders: 23 categorical columns
✓ Loaded feature_columns: 49 features

All models loaded successfully!


## 3. Load Training Data (for reference and evaluation)

In [ ]:
data_train_dir = Path('.data/data_train')

if data_train_dir.exists():
    print("Loading training data from .data/data_train/")
    print("=" * 70)
    
    X = joblib.load(data_train_dir / 'X.joblib')
    y_etiquette = joblib.load(data_train_dir / 'y_etiquette.joblib')
    y_value = joblib.load(data_train_dir / 'y_value.joblib')
    
    print(f"X shape: {X.shape}")
    print(f"y_etiquette shape: {y_etiquette.shape}")
    print(f"y_value shape: {y_value.shape}")
    print(f"Data loaded successfully")
else:
    print("Training data not found in .data/data_train/")
    X = None
    y_etiquette = None
    y_value = None

Loading training data from .data/data_train/
✓ X shape: (166680, 49)
✓ y_etiquette shape: (166680,)
✓ y_value shape: (166680,)
✓ Data loaded successfully


## 4. Define Preprocessing Function for Raw Input

In [ ]:
def preprocess_raw_input(raw_data, feature_columns, label_encoders, X_train=None):
    if isinstance(raw_data, dict):
        raw_data = pd.DataFrame([raw_data])
    
    report = {'status': 'OK', 'warnings': [], 'errors': []}
    
    # Check for extra columns - remove them
    extra_cols = set(raw_data.columns) - set(feature_columns)
    if extra_cols:
        report['warnings'].append(f"Ignoring {len(extra_cols)} extra columns")
        raw_data = raw_data.drop(columns=list(extra_cols))
    
    # Select and reorder features
    df = raw_data[feature_columns].copy()
    
    # Encode categorical columns
    for col in df.columns:
        if col in label_encoders:
            df[col] = df[col].astype(str)
            known = set(label_encoders[col].classes_)
            unknown_mask = ~df[col].isin(known)
            
            if unknown_mask.any():
                # Map unknown to most common class
                df.loc[unknown_mask, col] = label_encoders[col].classes_[0]
                report['warnings'].append(f"Unknown values in '{col}' mapped to '{label_encoders[col].classes_[0]}'")
            
            df[col] = label_encoders[col].transform(df[col])
        else:
            # For non-encoded columns, ensure numeric
            if df[col].dtype == 'object':
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].isna().any():
                        df[col] = df[col].fillna(X_train[col].median() if X_train is not None else 0)
                except:
                    df[col] = 0
    
    if report['warnings']:
        report['status'] = 'WARNING'
    
    return df, report

print("Preprocessing function defined")

✓ Preprocessing function defined


## 5. Example: Make Predictions

In [ ]:
if X is not None:
    print("\nExample: Predictions on training input")
    print("=" * 70)
    
    # Pick 3 random samples
    indices = np.random.choice(len(X), 3, replace=False)
    
    for idx, sample_idx in enumerate(indices, 1):
        # Get raw sample
        sample = X.iloc[sample_idx].to_dict()
        
        # Preprocess
        processed, report = preprocess_raw_input(sample, feature_columns, label_encoders, X)
        
        # Make predictions
        pred_et = rf_etiquette.predict(processed)[0]
        pred_proba = rf_etiquette.predict_proba(processed)[0]
        pred_val = rf_value.predict(processed)[0]
        
        print(f"\nSample {idx}:")
        print(f"  Input columns: {len(sample)} (original)")
        if report['status'] == 'WARNING':
            print(f"  Preprocessing: {report['warnings'][0] if report['warnings'] else 'OK'}")
        
        print(f"  Predicted Etiquette: {pred_et}")
        print(f"  Confidence: {max(pred_proba):.1%}")
        print(f"  Energy Value: {pred_val:.2f} kWh/m²")
else:
    print("Training data not available for predictions")


Example: Predictions on training input

Sample 1:
  Input columns: 49 (original)
  Predicted Etiquette: A
  Confidence: 97.2%
  Energy Value: 30.64 kWh/m²

Sample 2:
  Input columns: 49 (original)
  Predicted Etiquette: A
  Confidence: 98.6%
  Energy Value: 32.11 kWh/m²

Sample 3:
  Input columns: 49 (original)
  Predicted Etiquette: A
  Confidence: 97.7%
  Energy Value: 28.37 kWh/m²
